# MaizeCare: maize leaf disease and severity

This notebook walks through the whole project **without training anything**. It reads the trained models and
their results from Google Drive and runs the same code as the training scripts and the web app (the `maize`
package, installed from GitHub), so every number here comes from the real pipeline.

1. **Data**: what the data pipeline (`maize-prepare`) kept, near-copy groups, the train / val / test split
2. **Exploratory data analysis**: class balance, photo sizes, colour, brightness, t-SNE
3. **Stage 1, the classifier**: which disease? Four backbones compared, MobileNetV2 chosen
4. **The leaf mask**: GrabCut grown from the middle of the photo
5. **Stage 2, the autoencoders**: how severe? Trained on healthy leaves only; the error is the severity
6. **Test results** against the proposal's targets
7. **The full pipeline** on test photos, and on your own photo

A CPU runtime is enough. Runtime -> Run all takes a few minutes, most of it reading photos from Drive.

In [ ]:
import os
import sys

os.environ.setdefault("KERAS_BACKEND", "torch")  # the backend the models were trained with; set before Keras loads
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q "maize[data,serve] @ git+https://github.com/Kos-en/maizecare.git@mask-and-severity"

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from IPython.display import Image, display
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import f1_score, roc_auc_score, roc_curve

from maize.autoencoder import Autoencoder, chosen, compare
from maize.classifier import build, expected_calibration_error
from maize.evaluate import onnx_batch_size, symptom_pixels
from maize.pipeline import HEALTHY, load, load_manifest, near_pairs, select
from maize.preprocess import AE_SIZE, CLS_SIZE, MASK_OK, decode, error_map, leaf_fraction, mask_background, prepare
from maize.serve import Scorer, onnx_runner

ROOT = Path(os.environ.get("MAIZE_ROOT", "/content/drive/MyDrive"))  # manifest.csv, the photos, the image caches
MODELS = ROOT / "maize_models"
CLASSIFIER = MODELS / "clf_mobilenet_v2"  # the chosen classifier
RUN = MODELS / "v2"                       # the autoencoder run with the GrabCut leaf mask

# The autoencoder maize-evaluate picks (best AUROC on its weakest disease, on val), if it has been evaluated.
best, _ = chosen(RUN)
evaluated = [f for f in sorted(RUN.glob("ae_*")) if (f / "metrics.json").exists()]
AE = best if best in evaluated else evaluated[0]
if AE != best:
    print(f"Note: maize-evaluate now picks {best.name}, but only {AE.name} has been evaluated.")
print("classifier:", CLASSIFIER.name, "| autoencoder:", AE.name)

# One colour per class everywhere in this notebook (colour-blind checked; scatter plots also use shapes).
COLOURS = {"Healthy": "#1baf7a", "MLB": "#2a78d6", "MLND": "#eda100", "MSV": "#e34948"}
MARKERS = {"Healthy": "o", "MLB": "s", "MLND": "^", "MSV": "D"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "figure.dpi": 100})

## 1. Data

Every photo goes through the data pipeline first (`maize-prepare`):

1. **File check**: it must open as a JPEG/PNG and its shortest side must be at least 50 px.
2. **Near copies**: two photos within 6 bits on **both** pHash and aHash are near copies; chains of near copies
   form one group.
3. **Mixed labels**: a group holding two classes is dropped whole; exact copies keep one.
4. **Split by group, stratified by class**: 70 / 15 / 15, so the same leaf can never be in both training and test.

In [ ]:
df = load_manifest(ROOT)
usable = select(df)
LABELS = sorted(usable["label"].unique())
print(f"{len(df)} photos in the manifest, {len(usable)} usable")
display(df.groupby(["label", "status"]).size().unstack(fill_value=0))

In [ ]:
splits = usable.groupby(["label", "split"]).size().unstack(fill_value=0)[["train", "val", "test"]]
splits["groups"] = usable.groupby("label")["group"].nunique()
display(splits)

shares = splits[["train", "val", "test"]].div(splits[["train", "val", "test"]].sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(7, 2.8))
left = np.zeros(len(shares))
for split, shade in zip(["train", "val", "test"], ["#2a78d6", "#86b6ef", "#c9def7"]):
    ax.barh(shares.index, shares[split], left=left, color=shade, edgecolor="white", linewidth=2, label=split)
    left += shares[split]
ax.axvline(0.70, color="#555", ls=":", lw=1)
ax.axvline(0.85, color="#555", ls=":", lw=1)
ax.set(xlim=(0, 1), xlabel="share of the class's photos", title="Split per class (dotted: the 70 / 15 / 15 target)")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False)
ax.grid(False)
plt.show()

**Near-copy groups and the leak check.** Most photos are alone in their group; a group of several is the same
leaf shot more than once. Recomputing every near-copy pair here, none may cross a split.

In [ ]:
pairs = near_pairs(usable)  # the same comparison maize-prepare makes (both hashes within 6 bits)
split_of = usable["split"].to_numpy()
leaks = int((split_of[pairs[:, 0]] != split_of[pairs[:, 1]]).sum())
print(f"{len(pairs):,} near-copy pairs among usable photos; {leaks} cross a split (must be 0)")

sizes = usable.groupby("group").size()
fig, ax = plt.subplots(figsize=(7, 3))
counts = sizes.value_counts().sort_index()
ax.bar(counts.index, counts.values, color="#2a78d6", width=0.8)
ax.set(yscale="log", xlabel="photos in the group", ylabel="number of groups (log)",
       title=f"Near-copy group sizes ({len(sizes):,} groups, largest {sizes.max()})")
plt.show()

## 2. Exploratory data analysis

On a random sample of the usable photos per class, read at full size from Drive (raise `SAMPLE` for more, at the
cost of time). As in the earlier study: class balance, image resolution, colour channels, brightness and a t-SNE
map of the raw pixels. The question behind each one: could a model tell the classes apart for the wrong
reason (photo size, lighting, camera) instead of the disease?

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
counts = usable["label"].value_counts().reindex(LABELS)
ax.pie(counts, labels=[f"{c}\n{n:,}" for c, n in counts.items()], colors=[COLOURS[c] for c in LABELS],
       autopct="%1.0f%%", startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2},
       textprops={"color": "#222"})
ax.set_title("Usable photos per class")
plt.show()

In [ ]:
SAMPLE = 150   # photos per class
THUMB = 64     # t-SNE input size


def photo_stats(path: str) -> dict:
    """Size, brightness, colour histograms and a small thumbnail of one original photo."""
    bgr = cv2.imread(str(ROOT / path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    histograms = [cv2.calcHist([rgb], [c], None, [256], [0, 256]).ravel() for c in range(3)]
    return {"width": rgb.shape[1], "height": rgb.shape[0],
            "brightness": float(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).mean()),
            "histograms": np.stack(histograms) / (rgb.shape[0] * rgb.shape[1]),
            "thumb": cv2.resize(rgb, (THUMB, THUMB), interpolation=cv2.INTER_AREA)}


sample = usable.groupby("label").sample(frac=1, random_state=0).groupby("label").head(SAMPLE)
with ThreadPoolExecutor(16) as pool:  # Drive reads are slow one by one, fast in parallel
    stats = pd.DataFrame(list(pool.map(photo_stats, sample["path"])), index=sample.index)
eda = sample[["path", "label"]].join(stats)
print(f"{len(eda)} photos read")

### Image resolution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in LABELS:
    part = eda[eda["label"] == label]
    ax.scatter(part["width"], part["height"], s=22, alpha=0.6, color=COLOURS[label], marker=MARKERS[label],
               edgecolor="white", linewidth=0.5, label=label)
ax.set(xlabel="width (px)", ylabel="height (px)", title="Photo size per class")
ax.legend(title="class")
plt.show()
display(eda.groupby("label")[["width", "height"]].agg(["mean", "min", "max"]).round(0).astype(int))

Every photo is scaled so its longest side is 224 px (128 px for the autoencoder) and padded to a square, so
size differences between classes do not reach the models as different shapes. They can still differ in
sharpness, which is one reason the classifier trains with blur augmentation.

### Colour channels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for channel, (ax, name) in enumerate(zip(axes, ["Red", "Green", "Blue"])):
    for label in LABELS:
        mean_histogram = np.mean(np.stack(eda.loc[eda["label"] == label, "histograms"]), axis=0)[channel]
        ax.plot(mean_histogram, color=COLOURS[label], lw=2, label=label)
    ax.set(title=f"{name} channel", xlabel="pixel value (0 = dark, 255 = bright)", xlim=(0, 255))
axes[0].set_ylabel("share of pixels")
axes[0].legend(title="class")
plt.suptitle("Average colour distribution per class", y=1.02)
plt.show()

Heavily overlapping curves mean the classes were photographed under similar light, so colour alone does not give
the class away. Where a class's curve sits apart (more red and less green), that is the disease itself:
yellowing and browning move pixels from green towards red.

### Brightness

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
parts = ax.boxplot([eda.loc[eda["label"] == c, "brightness"] for c in LABELS], tick_labels=LABELS,
                   patch_artist=True, widths=0.5, medianprops={"color": "#222"})
for box, label in zip(parts["boxes"], LABELS):
    box.set(facecolor=COLOURS[label], alpha=0.75)
ax.set(ylabel="mean grey level (0 = black, 255 = white)", title="Brightness per class")
plt.show()
display(eda.groupby("label")["brightness"].describe()[["mean", "50%", "std"]].round(1))

### t-SNE of the raw pixels

Each photo shrunk to 64 x 64, flattened to 12,288 numbers and scaled to 0-1, reduced to 50 dimensions with PCA,
then laid out in 2-D by t-SNE so that similar photos end up close together. Clear clusters would mean the classes
differ in plain pixel layout; overlap means a model has to learn the symptoms themselves.

In [ ]:
pixels = np.stack(eda["thumb"]).reshape(len(eda), -1) / 255.0
embedded = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=42).fit_transform(
    PCA(n_components=min(50, len(eda) - 1), random_state=42).fit_transform(pixels))

fig, ax = plt.subplots(figsize=(7, 6))
for label in LABELS:
    keep = (eda["label"] == label).to_numpy()
    ax.scatter(*embedded[keep].T, s=24, alpha=0.7, color=COLOURS[label], marker=MARKERS[label],
               edgecolor="white", linewidth=0.5, label=label)
ax.set(title="t-SNE of raw pixels (64 x 64)", xticks=[], yticks=[])
ax.legend(title="class")
plt.show()

## 3. Stage 1: the classifier

Four ImageNet backbones, transfer learning (train the head, then fine-tune the top 40 layers), all on the same
split. Target (a): within 3 points of ResNet50 and at least 3x smaller. MobileNetV2 is chosen.

In [ ]:
def classifier_row(folder: Path) -> dict:
    info = json.loads((folder / "classifier.json").read_text())
    return {"backbone": info["backbone"], "val_macro_f1": info["val"]["macro_f1"],
            "test_macro_f1": info["test"]["macro_f1"], "test_ece": info["test"]["ece"],
            "size_mb": (folder / "classifier.onnx").stat().st_size / 1e6}


classifiers = pd.DataFrame([classifier_row(f) for f in sorted(MODELS.glob("clf_*"))])
classifiers = classifiers.sort_values("val_macro_f1", ascending=False, ignore_index=True)
display(classifiers.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
y = np.arange(len(classifiers))
axes[0].barh(y - 0.2, classifiers["val_macro_f1"], 0.38, color="#86b6ef", label="val")
axes[0].barh(y + 0.2, classifiers["test_macro_f1"], 0.38, color="#2a78d6", label="test")
axes[0].set(yticks=y, yticklabels=classifiers["backbone"], xlim=(0, 1), xlabel="macro-F1", title="Accuracy")
axes[0].legend(loc="lower right")
axes[1].barh(y, classifiers["size_mb"], 0.6, color="#555")
for i, mb in enumerate(classifiers["size_mb"]):
    axes[1].text(mb, i, f" {mb:.1f} MB", va="center")
axes[1].set(yticks=y, yticklabels=classifiers["backbone"], xlabel="ONNX file size (MB)", title="Size")
for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

**The chosen model.** The architecture rebuilt from the code (no weights needed to show it), then its training
curves: phase 1 trains only the new head, phase 2 fine-tunes the top of the backbone at a lower learning rate.

In [ ]:
info = json.loads((CLASSIFIER / "classifier.json").read_text())
labels = info["labels"]
model, _ = build(info["backbone"], len(labels), weights=None)
model.summary()

In [ ]:
history = pd.read_csv(CLASSIFIER / "history.csv")
history["epoch"] = np.arange(1, len(history) + 1)
fine_tune_starts = history.loc[history["phase"] == "finetune", "epoch"].min()
curves = [("loss", "val_loss", "loss"), ("accuracy", "val_accuracy", "accuracy")]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (train_col, val_col, name) in zip(axes, curves):
    ax.plot(history["epoch"], history[train_col], color="#2a78d6", lw=2, label="train")
    ax.plot(history["epoch"], history[val_col], color="#eb6834", lw=2, label="val")
    ax.set(title=name, xlabel="epoch")
    ax.legend()
if "val_macro_f1" in history:
    axes[2].plot(history["epoch"], history["val_macro_f1"], color="#eb6834", lw=2, marker="o")
    axes[2].set(title="val macro-F1 (best epoch kept)", xlabel="epoch")
for ax in axes:
    if pd.notna(fine_tune_starts):
        ax.axvline(fine_tune_starts - 0.5, color="#888", ls=":", lw=1)
plt.suptitle(f"{info['backbone']}: training (dotted line: fine-tuning starts)", y=1.03)
plt.show()

**On the test photos.** The confusion matrix and per-class scores were saved when the model was trained. Then the
exported ONNX model, the same file the web app runs, is run again on the test photos here, which also gives the
probabilities for the calibration and ROC plots.

In [ ]:
matrix = np.array(info["test"]["confusion_matrix"])
row_share = matrix / matrix.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(row_share, cmap="Greens", vmin=0, vmax=1)
ax.set(xticks=range(len(labels)), xticklabels=labels, yticks=range(len(labels)), yticklabels=labels,
       xlabel="predicted", ylabel="true", title=f"{info['backbone']}: test photos")
ax.grid(False)
for (true, predicted), n in np.ndenumerate(matrix):
    ax.text(predicted, true, f"{n}\n{row_share[true, predicted]:.0%}", ha="center", va="center",
            color="white" if row_share[true, predicted] > 0.6 else "#222")
plt.show()
display(pd.DataFrame(info["test"]["per_class"]).T.loc[labels, ["precision", "recall", "f1-score", "support"]].round(3))

In [ ]:
test = select(df, split="test")
test_images, test_masks = load(test, ROOT, CLS_SIZE)  # the cached 224 px inputs the classifier was tested on
y_test = test["label"].map(labels.index).to_numpy()

session = ort.InferenceSession(str(CLASSIFIER / "classifier.onnx"), providers=["CPUExecutionProvider"])
input_name, batch = session.get_inputs()[0].name, onnx_batch_size(session)


def classify(images: np.ndarray) -> np.ndarray:
    """Class probabilities from the exported classifier, in batches it accepts."""
    return np.concatenate([session.run(None, {input_name: images[i:i + batch].astype(np.float32)})[0]
                           for i in range(0, len(images), batch)])


start = time.perf_counter()
probs = classify(test_images)
per_photo_ms = 1000 * (time.perf_counter() - start) / len(test_images)
print(f"{len(test_images)} test photos, macro-F1 {f1_score(y_test, probs.argmax(1), average='macro'):.3f}, "
      f"{per_photo_ms:.1f} ms per photo on this CPU")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Calibration: is a 90 %-confident answer right 90 % of the time?
confidence, correct = probs.max(1), probs.argmax(1) == y_test
edges = np.linspace(0, 1, 11)
bin_of = np.clip(np.digitize(confidence, edges) - 1, 0, 9)
seen = [b for b in range(10) if (bin_of == b).any()]
axes[0].plot([0, 1], [0, 1], color="#888", ls=":", lw=1, label="perfectly calibrated")
axes[0].plot([confidence[bin_of == b].mean() for b in seen], [correct[bin_of == b].mean() for b in seen],
             color="#2a78d6", lw=2, marker="o", label="classifier")
axes[0].axvline(info["min_confidence"], color="#eb6834", ls="--", lw=1, label="app asks for a retake below")
axes[0].set(xlabel="confidence", ylabel="share correct", xlim=(0, 1), ylim=(0, 1.02),
            title=f"Calibration (ECE {expected_calibration_error(probs, y_test):.3f})")
axes[0].legend(loc="upper left")

# One-vs-rest ROC per class
for i, label in enumerate(labels):
    if 0 < (y_test == i).sum() < len(y_test):
        fpr, tpr, _ = roc_curve(y_test == i, probs[:, i])
        auc = roc_auc_score(y_test == i, probs[:, i])
        axes[1].plot(fpr, tpr, color=COLOURS.get(label, "#555"), lw=2, label=f"{label} (AUC {auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="#888", ls=":", lw=1)
axes[1].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC, each class vs the rest")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()
sure = confidence >= info["min_confidence"]
print(f"{1 - sure.mean():.1%} of test photos fall below the retake threshold ({info['min_confidence']})"
      + (f"; accuracy on the rest: {correct[sure].mean():.1%}" if sure.any() else ""))

**Field robustness.** Farmers' photos are often out of focus or dark. The test photos are degraded here (a check
in this notebook only, nothing is retrained) and classified again. The classifier was trained with blur,
brightness and noise augmentation, so it should lose little.

In [ ]:
def blurred(images: np.ndarray) -> np.ndarray:
    return np.stack([cv2.GaussianBlur(image, (0, 0), 2.0) for image in images])  # out of focus


def low_light(images: np.ndarray) -> np.ndarray:
    noise = np.random.default_rng(0).normal(0, 8, images.shape)
    return np.clip(images * 0.45 + noise, 0, 255).astype(np.uint8)  # about half as bright, with sensor noise


variants = {"clean": test_images, "blurred": blurred(test_images), "low light": low_light(test_images)}
predicted = {name: classify(images).argmax(1) for name, images in variants.items()}
robustness = pd.DataFrame({name: {"accuracy": (pred == y_test).mean(),
                                  "macro_f1": f1_score(y_test, pred, average="macro")}
                           for name, pred in predicted.items()}).T
display(robustness.round(3))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), gridspec_kw={"width_ratios": [1, 1, 1, 1.6]})
for ax, (name, images) in zip(axes, variants.items()):
    ax.imshow(images[0])
    ax.set_title(name)
    ax.axis("off")
axes[3].bar(robustness.index, robustness["macro_f1"], color=["#2a78d6", "#86b6ef", "#86b6ef"], width=0.6)
for i, f1 in enumerate(robustness["macro_f1"]):
    axes[3].text(i, f1, f"{f1:.3f}", ha="center", va="bottom")
axes[3].set(ylim=(0, 1.08), ylabel="macro-F1", title="Test photos, degraded")
plt.tight_layout()
plt.show()

## 4. The leaf mask

The autoencoder must only look at the photographed leaf. GrabCut learns the leaf's colours from the middle of the
photo (the farmer aims at the leaf) and the background's colours from the rest; the piece holding the middle is kept
and holes (lesions) are filled. An earlier colour rule (Excess Green) kept weeds and dropped yellowed leaves;
replacing it raised the val AUROC from 0.65 to 0.86. Masks covering under 10% or over 95% of the photo are flagged
as doubtful and left out of severity scoring.

In [ ]:
def read_rgb(path: str) -> np.ndarray:
    return cv2.cvtColor(cv2.imread(str(ROOT / path)), cv2.COLOR_BGR2RGB)


def outline(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """The photo dimmed outside the leaf, with the mask edge in magenta."""
    shown = (image * 0.35).astype(np.uint8)
    shown[mask == 1] = image[mask == 1]
    edges, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return cv2.drawContours(shown, edges, -1, (255, 0, 255), 2)


examples = test.groupby("label").sample(2, random_state=1)
fig, axes = plt.subplots(2, len(LABELS), figsize=(3.5 * len(LABELS), 7), squeeze=False)
for ax, (_, leaf) in zip(axes.T.ravel(), examples.iterrows()):  # one column per class
    image, mask = prepare(read_rgb(leaf["path"]), CLS_SIZE)
    ax.imshow(outline(image, mask))
    ax.set_title(f"{leaf['label']}: leaf {leaf_fraction(mask):.0%} of photo")
    ax.axis("off")
plt.show()

In [ ]:
fractions = test_masks.mean(axis=(1, 2))
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0, 1, 26)
for label in LABELS:
    ax.hist(fractions[(test["label"] == label).to_numpy()], bins=bins, histtype="step", lw=2,
            color=COLOURS[label], label=label)
for bound in MASK_OK:
    ax.axvline(bound, color="#888", ls=":", lw=1)
ax.set(xlabel="share of the photo that is leaf", ylabel="test photos",
       title="Leaf mask size (outside the dotted lines: doubtful, left out)")
ax.legend()
plt.show()

## 5. Stage 2: the autoencoders

A convolutional autoencoder and a VAE of the same shape are trained on **healthy leaves only**, inside the leaf
mask. A diseased leaf rebuilds badly, so its reconstruction error is the severity signal: no severity labels are
needed. Encoder: 4 stride-2 blocks (32 -> 256 filters, 128 px -> 8 px) and a 1x1 layer down to 32 channels, 24x
compression, so the model cannot just copy its input.

In [ ]:
Autoencoder.load(AE).model.summary()  # the chosen autoencoder, loaded from Drive

In [ ]:
folders = sorted(RUN.glob("ae_*"))
fig, axes = plt.subplots(1, len(folders), figsize=(6 * len(folders), 3.5), squeeze=False)
for ax, folder in zip(axes[0], folders):
    history = pd.read_csv(folder / "history.csv")
    epochs = np.arange(1, len(history) + 1)
    ax.plot(epochs, history["loss"], color="#2a78d6", lw=2, label="train (healthy)")
    ax.plot(epochs, history["val_loss"], color="#eb6834", lw=2, label="val (healthy)")
    ax.set(title=folder.name, xlabel="epoch", ylabel="loss")
    ax.legend()
plt.suptitle("Autoencoder training (early stopping keeps the best val epoch)", y=1.03)
plt.show()

**Choosing one, on val.** Each autoencoder is scored with the plain error and a smoothed one (colour and tone, not
fine texture). The pair with the best AUROC on its **weakest** disease is chosen, because target (b) is per disease.
This is the same table `maize-train-autoencoder` writes.

In [ ]:
display(compare(RUN).drop(columns="healthy_median_error").round(3))

## 6. Test results against the targets

The test photos were used once, after every choice was made on val.

* **(b)** AUROC (healthy vs each disease) >= 0.80
* **(c)** Spearman with the symptomatic area A(x) >= 0.40 for MLND and MLB, and better than both baselines.
  A(x) = % of the leaf in a symptom colour range (yellowing, brown lesions, dead tissue), like Leaf Doctor / ASSESS.

In [ ]:
metrics = json.loads((AE / "metrics.json").read_text())
results = pd.DataFrame({
    disease: {"AUROC": r["auroc_vs_healthy"], "Spearman vs A(x)": r["spearman_vs_area"],
              **{f"baseline: {name}": value for name, value in metrics["baselines"][disease].items()}}
    for disease, r in metrics["diseases"].items()}).T
results["(b) AUROC >= 0.80"] = results["AUROC"] >= 0.80
results["(c) Spearman >= 0.40"] = results["Spearman vs A(x)"] >= 0.40
display(results.round(3))
print(f"error measured: {metrics['error']}")
print(f"healthy test leaves scored as unusually high: {metrics['false_high_rate']:.1%}")
print("doubtful masks:", {label: f"{share:.0%}" for label, share in metrics["doubtful_mask_share"].items()})

In [ ]:
scores = pd.read_csv(AE / "test_scores.csv")
healthy_errors = scores.loc[scores["label"] == HEALTHY, "error"]
diseases = sorted(set(scores["label"]) - {HEALTHY})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for disease in diseases:
    errors = scores.loc[scores["label"] == disease, "error"]
    is_disease = np.r_[np.zeros(len(healthy_errors)), np.ones(len(errors))]
    fpr, tpr, _ = roc_curve(is_disease, np.r_[healthy_errors, errors])
    axes[0].plot(fpr, tpr, color=COLOURS.get(disease, "#555"), lw=2,
                 label=f"{disease} (AUROC {metrics['diseases'][disease]['auroc_vs_healthy']:.2f})")
axes[0].plot([0, 1], [0, 1], color="#888", ls=":", lw=1)
axes[0].set(xlabel="healthy leaves flagged", ylabel="diseased leaves caught",
            title="Reconstruction error: healthy vs each disease")
axes[0].legend(loc="lower right")

methods = {"autoencoder": "Spearman vs A(x)", "colour distance": "baseline: colour_distance",
           "classifier 1 - P(healthy)": "baseline: classifier_1_minus_p_healthy"}
methods = {name: column for name, column in methods.items() if column in results}
x = np.arange(len(diseases))
width = 0.8 / len(methods)
for i, ((name, column), shade) in enumerate(zip(methods.items(), ["#2a78d6", "#aaa", "#666"])):
    axes[1].bar(x + (i - (len(methods) - 1) / 2) * width, results.loc[diseases, column], width * 0.9,
                color=shade, label=name)
axes[1].axhline(0.40, color="#eb6834", ls="--", lw=1, label="target 0.40")
axes[1].axhline(0, color="#888", lw=0.8)
axes[1].set(xticks=x, xticklabels=diseases, ylabel="Spearman with A(x)", title="Severity ranking vs baselines")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(diseases), figsize=(5 * len(diseases), 4), squeeze=False, sharey=True)
for ax, disease in zip(axes[0], diseases):
    part = scores[scores["label"] == disease]
    ax.scatter(part["area"], part["score"], s=18, alpha=0.6, color=COLOURS.get(disease, "#555"),
               marker=MARKERS.get(disease, "o"), edgecolor="white", linewidth=0.5)
    ax.set(title=f"{disease} (Spearman {metrics['diseases'][disease]['spearman_vs_area']:.2f})",
           xlabel="symptomatic area A(x), % of leaf")
axes[0][0].set_ylabel("severity score (0-1)")
plt.suptitle("Severity score against symptomatic area, test leaves", y=1.03)
plt.show()

**What A(x) counts.** The reference the score is checked against: leaf pixels in a symptom colour range, shown in
magenta on one test leaf per disease.

In [ ]:
fig, axes = plt.subplots(2, len(diseases), figsize=(4 * len(diseases), 8), squeeze=False)
for column, disease in enumerate(diseases):
    i = int(np.flatnonzero((test["label"] == disease).to_numpy())[0])
    image, mask = test_images[i], test_masks[i]
    symptoms = symptom_pixels(image) & (mask > 0)
    marked = image.copy()
    marked[symptoms] = (255, 0, 255)
    axes[0][column].imshow(outline(image, mask))
    axes[0][column].set_title(disease)
    axes[1][column].imshow(marked)
    axes[1][column].set_title(f"A(x) = {100 * symptoms.sum() / max(mask.sum(), 1):.1f}% of the leaf")
    for ax in axes[:, column]:
        ax.axis("off")
plt.show()

Saved by `maize-evaluate`: the error per class on test, and heat maps of where on the leaf the error sits
(input inside the mask, reconstruction, error).

In [ ]:
display(Image(str(AE / "error_by_class.png")))
display(Image(str(AE / "heatmaps.png")))

## 7. The full pipeline

Exactly what the web app does for an uploaded photo (`maize.serve.Scorer`, the same ONNX files):

1. the classifier names the disease from the whole photo (below 60% confidence the app asks for a retake);
2. the leaf mask is found; the autoencoder rebuilds the leaf;
3. its error inside the mask becomes a 0-1 severity score (0 = like a typical healthy leaf, 1 = among the worst
   diseased leaves seen), which the app shows as mild (< 0.35), moderate (< 0.70) or severe.

In [ ]:
classes = json.loads((CLASSIFIER / "classifier.json").read_text())
calibration = json.loads((AE / "calibration.json").read_text())
scorer = Scorer(onnx_runner(CLASSIFIER / "classifier.onnx"), onnx_runner(AE / "model.onnx"), classes, calibration)


def band(label: str, score: float) -> str:
    """The grade the app shows (frontend/src/advice.ts)."""
    if label == HEALTHY:
        return "healthy"
    return "mild" if score < 0.35 else "moderate" if score < 0.7 else "severe"


def show(rgb: np.ndarray, title: str = "") -> dict:
    """Score one photo like the app, and draw every step."""
    result = scorer.score(rgb)
    image, mask = prepare(rgb, AE_SIZE)
    masked = mask_background(image[None], mask[None])
    recon = scorer.reconstruct(masked.astype(np.float32))
    error = error_map(image[None], recon, mask[None], calibration.get("smooth_sigma", 0.0))[0]

    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    pictures = [(outline(*prepare(rgb, CLS_SIZE)), "photo + leaf mask"), (masked[0], "autoencoder input"),
                (recon[0], "reconstruction"), (error, "error (brighter = worse)")]
    for ax, (picture, name) in zip(axes, pictures):
        ax.imshow(picture, cmap="hot")
        ax.set_title(name)
        ax.axis("off")
    retake = "" if result["confident"] else ", app asks for a retake"
    plt.suptitle(f"{title}  ->  {result['label']} ({result['confidence']:.0%}{retake}), "
                 f"severity {result['score']:.2f}: {band(result['label'], result['score'])}", y=1.02)
    plt.show()
    return result


for _, leaf in test.groupby("label").sample(1, random_state=7).iterrows():
    show(read_rgb(leaf["path"]), f"true: {leaf['label']}")

In [ ]:
timed = [read_rgb(p) for p in test["path"].sample(min(20, len(test)), random_state=0)]
start = time.perf_counter()
for rgb in timed:
    scorer.score(rgb)
print(f"Whole pipeline (classify, mask, reconstruct, score): {1000 * (time.perf_counter() - start) / len(timed):.0f} "
      "ms per photo on this CPU")

Try your own photo (centre the leaf in the picture):

In [ ]:
if IN_COLAB:
    from google.colab import files

    for name, data in files.upload().items():
        show(decode(data), name)

## Limitations and next steps

* Detection (b) is met; severity ranking (c) is partial: the autoencoder follows streaks and lesions (MSV, MLB) better
  than broad yellowing (MLND), which it partly rebuilds.
* A(x) is a colour rule, not a person's judgement: it is a reasonable reference, not ground truth.
* Some photos in the Healthy folder are not maize (e.g. bean leaves): to be cleaned.
* The robustness check above uses synthetic blur and darkening; real field photos would be the true test.
* Next: a colour-reconstruction autoencoder (grey leaf in, colour out) for MLND, human by-eye severity ratings,
  confidence intervals, and results per data source.